# Flight Route Analysis

This notebook analyzes flight route patterns and characteristics in the Norwegian aviation dataset.

# !!!NB Deprecated, we dont have nessecary data for this!!!

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [28]:
data_path = "../data/historical_flights.csv"
df = pd.read_csv(data_path)


In [35]:
# Create daily flight sequence number for each plane
df['std'] = pd.to_datetime(df['std'])
df['flight_date'] = df['std'].dt.date

# Sort by flight_id, date, and departure time, then create sequence number
df_sorted = df.sort_values(['flight_id', 'flight_date', 'std'])
df_sorted['daily_flight_number'] = df_sorted.groupby(['flight_id', 'flight_date']).cumcount() + 1

# Merge back to original dataframe
df = df.merge(
    df_sorted[['flight_id', 'flight_date', 'std', 'daily_flight_number']], 
    on=['flight_id', 'flight_date', 'std'], 
    how='left'
)

# Check the results
df_sorted[['flight_id', 'flight_date', 'std', 'daily_flight_number', 'dep_airport', 'arr_airport']].head(30)

,flight_id,flight_date,std,daily_flight_number,dep_airport,arr_airport
324825,1SK800,2023-10-25,2023-10-25 15:00:00,1,OSL,HOV
324828,1SK800,2023-10-26,2023-10-26 15:00:00,1,OSL,HOV
359066,1SK903,2024-08-27,2024-08-27 07:10:00,1,BVG,OSL
375941,1SK903,2024-09-17,2024-09-17 05:35:00,1,BVG,OSL
359067,1SK905,2024-08-27,2024-08-27 07:10:00,1,HAA,OSL
375942,1SK905,2024-09-17,2024-09-17 05:35:00,1,HAA,OSL
361637,1SK934,2024-06-09,2024-06-09 14:00:00,1,OSL,MEH
349264,1SK936,2024-05-13,2024-05-13 10:15:00,1,OSL,MJF
356450,1SK936,2024-08-22,2024-08-22 10:15:00,1,OSL,MJF
359063,1SK936,2024-08-27,2024-08-27 10:15:00,1,OSL,MJF


In [31]:
# Clean function version
def add_daily_flight_sequence(df, flight_id_col='flight_id', departure_time_col='std'):
    df = df.copy()
    
    # Convert to datetime and extract date
    df[departure_time_col] = pd.to_datetime(df[departure_time_col])
    df['flight_date'] = df[departure_time_col].dt.date
    
    # Sort and create sequence number
    df = df.sort_values([flight_id_col, 'flight_date', departure_time_col])
    df['daily_flight_number'] = df.groupby([flight_id_col, 'flight_date']).cumcount() + 1
    
    return df

# Apply to fresh data (reload to avoid duplicate columns)
df = pd.read_csv("../data/historical_flights.csv")
df_with_sequence = add_daily_flight_sequence(df)

# Show results
print(f"Max flights per day for a single plane: {df_with_sequence['daily_flight_number'].max()}")
print(f"Average flights per day per plane: {df_with_sequence['daily_flight_number'].mean():.2f}")
df_with_sequence[['flight_id', 'flight_date', 'std', 'daily_flight_number', 'dep_airport', 'arr_airport']].head()

Max flights per day for a single plane: 10
Average flights per day per plane: 1.52


,flight_id,flight_date,std,daily_flight_number,dep_airport,arr_airport
324763,1SK800,2023-10-25,2023-10-25 15:00:00,1,OSL,HOV
324765,1SK800,2023-10-26,2023-10-26 15:00:00,1,OSL,HOV
359000,1SK903,2024-08-27,2024-08-27 07:10:00,1,BVG,OSL
375855,1SK903,2024-09-17,2024-09-17 05:35:00,1,BVG,OSL
359001,1SK905,2024-08-27,2024-08-27 07:10:00,1,HAA,OSL


In [32]:
# Fix the function - debug version
def add_daily_flight_sequence(df, flight_id_col='flight_id', departure_time_col='std'):
    df = df.copy()
    
    print(f"Input df shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Convert to datetime and extract date
    df[departure_time_col] = pd.to_datetime(df[departure_time_col])
    df['flight_date'] = df[departure_time_col].dt.date
    
    # Sort and create sequence number
    df_sorted = df.sort_values([flight_id_col, 'flight_date', departure_time_col])
    df_sorted['daily_flight_number'] = df_sorted.groupby([flight_id_col, 'flight_date']).cumcount() + 1
    
    print(f"After creating sequence, df_sorted shape: {df_sorted.shape}")
    print(f"daily_flight_number values: {df_sorted['daily_flight_number'].value_counts().head()}")
    
    # Use the sorted dataframe directly instead of merging
    return df_sorted

# Test the fixed function
df_with_sequence = add_daily_flight_sequence(df)
df_with_sequence[['flight_id', 'flight_date', 'std', 'daily_flight_number', 'dep_airport', 'arr_airport']].head()

Input df shape: (410442, 11)
Columns: ['flight_id', 'dep_airport', 'dep_airport_group', 'arr_airport', 'arr_airport_group', 'service_type', 'std', 'sta', 'cancelled', 'atd', 'ata']
After creating sequence, df_sorted shape: (410442, 13)
daily_flight_number values: daily_flight_number
1    267918
2     95860
3     29967
4      9692
5      5692
Name: count, dtype: int64


,flight_id,flight_date,std,daily_flight_number,dep_airport,arr_airport
324763,1SK800,2023-10-25,2023-10-25 15:00:00,1,OSL,HOV
324765,1SK800,2023-10-26,2023-10-26 15:00:00,1,OSL,HOV
359000,1SK903,2024-08-27,2024-08-27 07:10:00,1,BVG,OSL
375855,1SK903,2024-09-17,2024-09-17 05:35:00,1,BVG,OSL
359001,1SK905,2024-08-27,2024-08-27 07:10:00,1,HAA,OSL


In [33]:
# Debug: Check what columns we have and the data structure
print("Columns in df:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)

Columns in df: ['flight_id', 'dep_airport', 'dep_airport_group', 'arr_airport', 'arr_airport_group', 'service_type', 'std', 'sta', 'cancelled', 'atd', 'ata']

First few rows:
  flight_id dep_airport dep_airport_group arr_airport arr_airport_group  \
0     WF149         HOV                 B         OSL               NaN   
1     WF722         OSL               NaN         MJF                 D   
2     WF188         FDE                 A         OSL               NaN   
3     WF176         HOV                 B         OSL               NaN   
4     WF148         HOV                 B         OSL               NaN   

  service_type                         std                         sta  \
0            J  2018-01-02T16:40:00.000000  2018-01-02T17:15:00.000000   
1            J  2018-01-28T13:04:00.000000  2018-01-28T14:50:00.000000   
2            J  2018-04-07T07:10:00.000000  2018-04-07T08:10:00.000000   
3            J  2018-04-07T11:00:00.000000  2018-04-07T12:05:00.000000   
4   